## Partie 0 – Mise en place de l'environnement

### 4) Installer et importer seaborn, matplotlib et pandas

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
%matplotlib inline

### 5) Importer `mesures_capteurs.csv` dans le dataframe `df`

In [2]:
df = pd.read_csv("../data/mesures_capteurs.csv")
df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


### 6) Explorer le dataframe `df`

In [23]:
print(f"shape : {df.shape}")
print("--*-**-*--*-**-*---*------*--*-*---*-*-*---*---*-*-*----*--*--*------*-*---*----*--*---*-----*---*-----*-*-*")
print(f"info :")

df.info()
print("--*-**-*--*-**-*-----*---*---*----*---*----*--*-*------*-----*-----*------*-----*-----*-*-------*----*-*--")

print(f"Description :")
df.describe()

shape : (596, 9)
--*-**-*--*-**-*---*------*--*-*---*-*-*---*---*-*-*----*--*--*------*-*---*----*--*---*-----*---*-----*-*-*
info :
<class 'pandas.DataFrame'>
Index: 596 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     596 non-null    str    
 1   date_heure    596 non-null    str    
 2   id_capteur    596 non-null    str    
 3   batiment      596 non-null    str    
 4   temperature   590 non-null    float64
 5   humidite      591 non-null    float64
 6   pression      591 non-null    float64
 7   consommation  591 non-null    float64
 8   etat          596 non-null    str    
dtypes: float64(4), str(5)
memory usage: 46.6 KB
--*-**-*--*-**-*-----*---*---*----*---*----*--*-*------*-----*-----*------*-----*-----*-*-------*----*-*--
Description :


,temperature,humidite,pression,consommation
count,590.000000,591.000000,591.000000,591.000000
mean,24.899712,64.830152,1012.205415,208.482166
std,4.073456,10.787337,10.648486,72.047854
min,-18.500000,28.520000,850.000000,18.120000
25%,22.620000,58.060000,1006.780000,160.500000
50%,24.870000,65.350000,1012.820000,206.130000
75%,27.295000,71.590000,1017.820000,253.810000
max,58.700000,145.000000,1038.430000,875.000000


---
## Partie 1 – Gestion des doublons

### 1) Vérifier l'existence de doublons

In [9]:
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons : {nb_doublons}")

Nombre de doublons : 5


### 2) Supprimer les doublons puis vérifier la suppression

In [12]:
df = df.drop_duplicates()
print(f"Nombre de doublons après suppression : {df.duplicated().sum()}")
print(f"Nouvelle taille du dataframe : {df.shape}")

Nombre de doublons après suppression : 0
Nouvelle taille du dataframe : (600, 9)


---
## Partie 2 – Sélection de y (cible) et X (caractéristiques)

### 1) Définir la cible et les caractéristiques

`etat` est la **cible** (ce qu'on veut prédire) ; `temperature`, `humidite`, `pression` et `consommation` sont les **caractéristiques** (les variables explicatives à partir desquelles on prédit).

On retire d'abord les lignes où la cible `etat` est manquante — un modèle ne peut pas apprendre à partir d'un exemple dont on ne connaît pas la bonne réponse.

In [13]:
# On ne garde que les lignes où la cible est connue
df = df.dropna(subset=["etat"])

caracteristiques = ["temperature", "humidite", "pression", "consommation"]

X = df[caracteristiques]
y = df["etat"]

print("Shape de X :", X.shape)
print("Shape de y :", y.shape)

Shape de X : (596, 4)
Shape de y : (596,)


### 2) Afficher les cinq premières lignes de X et de y

In [19]:
X.head()

,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62


In [20]:
y.head()

0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: str

### 3) Quel est le type du problème de machine learning ?

C'est un problème de **classification supervisée multiclasse** :
- **Supervisée**, car on dispose déjà de la bonne réponse (`etat`) pour chaque exemple d'entraînement.
- **Classification**, car la cible à prédire est une **catégorie** (`OK`, `ALERTE`, `ERREUR`), pas une valeur numérique continue (ce qui serait de la régression).
- **Multiclasse**, car il y a plus de deux catégories possibles (contrairement à une classification binaire qui n'en aurait que deux).

---
## Partie 3 – Découpage Train/Test

On divise `X` (et `y`) en un ensemble d'entraînement et un ensemble de test, avec :
- **20% des données pour le test** (`test_size=0.2`)
- **Reproductibilité** du découpage (`random_state` fixé, pour retomber sur le même découpage à chaque exécution)
- **Mêmes proportions de classes** dans train et test qu'à l'origine (`stratify=y`) — essentiel ici vu le fort déséquilibre entre `OK`, `ALERTE` et `ERREUR`

In [24]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("\nRépartition des classes dans y_train :")
print(y_train.value_counts(normalize=True).round(3))
print("\nRépartition des classes dans y_test :")
print(y_test.value_counts(normalize=True).round(3))

X_train : (476, 4)
X_test  : (120, 4)

Répartition des classes dans y_train :
etat
OK        0.943
ALERTE    0.048
ERREUR    0.008
Name: proportion, dtype: float64

Répartition des classes dans y_test :
etat
OK        0.942
ALERTE    0.050
ERREUR    0.008
Name: proportion, dtype: float64
